In [1]:
import pandas as pd

amp_df = pd.read_csv(
    "clean_amp_dataset.csv"
)

print(amp_df.shape)

(8400, 6)


In [2]:
unknown_patterns = [
    "No hemolysis information",
    "Not available",
    "Not found"
]

mask = ~amp_df["Hemolytic_activity"].str.contains(
    "|".join(unknown_patterns),
    case=False,
    na=False
)

hemo_df = amp_df[mask].copy()

print("Hemolysis candidates:", hemo_df.shape)

Hemolysis candidates: (2029, 6)


In [3]:
hemo_df["Hemolytic_activity"].sample(
    50,
    random_state=42
).tolist()

['[Ref.33285267]5% hemolysis against mouse erythrocytes at 256 µM.',
 '[Ref.24105706] HC50>800 μM against human red blood cells',
 '[Ref.29130500] 0% hemolysis  at 64 μg/ml , 0.5% hemolysis at 128 μg/ml , 2% hemolysis  at 256 μg/ml , 13% hemolysis at 512 μg/ml against human red blood cells',
 'Human erythrocytes: <10% Hemolysis=50 µM',
 '[Ref.29307492] The peptide showed 1.7% hemolysis against human red blood cells at 50 μM. PBS was uesd as a negative control and 1% triton was used as a positive control for 100% lysis.',
 'Human erythrocytes: 50% Hemolysis>36 µM',
 '[Ref.31181304] 45% hemolysis at 12.5 μmol/L, 100% hemolysis at 50 μmol/L against human red blood cells',
 '[Ref.22467870] HD50>25 µM against human type A red blood cells.',
 'Human red blood cells (hRBCs): HC50>50 µM ',
 '[Ref.25312021]10.5% Hemolysis against Rat erythrocytes at 20 µM.',
 'Rat erythrocytes: IC50=86 μM',
 '[Ref:17272268] It has no hemolytic activity against rabbit red blood cells',
 '[Ref.15638808]Show hemol

In [4]:
import re
import numpy as np

def assign_hemo_label(text):

    t = str(text).lower()

    # Strong non-hemolytic patterns
    non_patterns = [
        "no hemolytic activity",
        "non-hemolytic",
        "does not induce substantial hemolysis",
        "not active",
        "little hemolytic activity",
        "low hemolytic activity",
        "negligible hemolytic activity",
        "0% hemolysis",
        "0.0% hemolysis",
        "1% hemolysis",
        "1.5% hemolysis",
        "0.3% hemolysis",
        "<5% hemolysis",
        "<10% hemolysis"
    ]

    for p in non_patterns:
        if p in t:
            return 0

    # Strong hemolytic patterns
    hemo_patterns = [
        "strong hemolytic activity",
        "show hemolytic activity",
        "has hemolytic activity",
        "100% hemolysis",
        "90% hemolysis",
        "85% hemolysis",
        "80% hemolysis",
        "70% hemolysis",
        "55% hemolysis",
        "50% hemolysis"
    ]

    for p in hemo_patterns:
        if p in t:
            return 1

    # HC50 / HD50 / IC50 / LC50 logic

    match = re.search(r'(hc50|hd50|ic50|lc50)\s*[=>]*\s*(\d+\.?\d*)', t)

    if match:

        value = float(match.group(2))

        # High value => safer
        if value >= 100:
            return 0

        # Low value => toxic
        if value <= 50:
            return 1

    return np.nan

In [5]:
hemo_df["hemo_label"] = hemo_df[
    "Hemolytic_activity"
].apply(assign_hemo_label)

print(
    hemo_df["hemo_label"].value_counts(
        dropna=False
    )
)

hemo_label
NaN    982
0.0    908
1.0    139
Name: count, dtype: int64


In [6]:
hemo_df[
    hemo_df["hemo_label"].isna()
]["Hemolytic_activity"].sample(
    50,
    random_state=123
).tolist()

['[Ref.2578459] Hemolysis: ED50=0.7±0.05 µg/ml. Histamine release: ED50=2.0±0.3 µg/ml against guinea pig erythrocytes.',
 'MRC-5: EC58=14 μM',
 'Human erythrocytes: 36 ± 1.7% Hemolysis=375 µM',
 'Human erythrocytes: 9 ± 0.4% Hemolysis=375 µM',
 '[Ref.15328096]EC50=3226000 nM against human erythrocytes',
 'Human erythrocytes: 12% Hemolysis=100 µg/ml',
 '[Ref.18000874]<5% hemolytic activity at 25-200 µg/ml against rat erythrocytes',
 '[Ref.23894079] MHC10>1280 μM against human red blood cells',
 'Human RBCs: less than 6% hemolysis at 100 μg/ml',
 'Human red blood cells: hemolysis 1.5%(50 μg/mL); hemolysis 8%(100 μg/mL)',
 '[Ref.30842436] < 8% hemolysis at 150 µM against human red blood cells',
 '[Ref.12199716] 0.02% hemolytic activity at 10 µg/mL, 0.38% hemolytic activity at 100 µg/mL against human red blood cells.',
 'Human erythrocytes: Hemolysis 0% (100 μM)',
 'Human red blood cells: MHC>334.4 μmol/L',
 '[Ref.9022710] LC=0.5 µM against human red blood cells. (LC: Lethal concentration)

In [7]:
hemo_text = hemo_df["Hemolytic_activity"].str.lower()

print("Contains HC50:",
      hemo_text.str.contains("hc50", na=False).sum())

print("Contains MHC:",
      hemo_text.str.contains("mhc", na=False).sum())

print("Contains IC50:",
      hemo_text.str.contains("ic50", na=False).sum())

print("Contains LC50:",
      hemo_text.str.contains("lc50", na=False).sum())

print("Contains % hemolysis:",
      hemo_text.str.contains("%", na=False).sum())

Contains HC50: 182
Contains MHC: 246
Contains IC50: 87
Contains LC50: 130
Contains % hemolysis: 977


In [8]:
import re

def extract_percent(text):

    text = str(text)

    vals = re.findall(r'(\d+\.?\d*)\s*%\s*hemo', text.lower())

    if len(vals) == 0:
        return None

    vals = [float(v) for v in vals]

    return max(vals)

hemo_df["max_hemo_percent"] = (
    hemo_df["Hemolytic_activity"]
    .apply(extract_percent)
)

print(
    hemo_df["max_hemo_percent"]
    .notna()
    .sum()
)

print(
    hemo_df["max_hemo_percent"]
    .describe()
)

887
count    887.000000
mean      25.098298
std       30.730885
min        0.000000
25%        3.000000
50%       10.000000
75%       50.000000
max      108.000000
Name: max_hemo_percent, dtype: float64


In [9]:
import re
import numpy as np

def final_hemo_label(text):

    t = str(text).lower()

    # -------- percentage based --------

    vals = re.findall(
        r'(\d+\.?\d*)\s*%\s*hemo',
        t
    )

    if len(vals) > 0:

        max_val = max(
            float(v) for v in vals
        )

        if max_val <= 10:
            return 0

        if max_val >= 20:
            return 1

    # -------- MHC --------

    mhc = re.search(
        r'mhc(?:10|5)?\s*[=><>]*\s*(\d+\.?\d*)',
        t
    )

    if mhc:

        value = float(
            mhc.group(1)
        )

        if value >= 128:
            return 0

        if value <= 32:
            return 1

    # -------- HC50 --------

    hc50 = re.search(
        r'hc50\s*[=><>]*\s*(\d+\.?\d*)',
        t
    )

    if hc50:

        value = float(
            hc50.group(1)
        )

        if value >= 100:
            return 0

        if value <= 50:
            return 1

    # -------- IC50 --------

    ic50 = re.search(
        r'ic50\s*[=><>]*\s*(\d+\.?\d*)',
        t
    )

    if ic50:

        value = float(
            ic50.group(1)
        )

        if value >= 100:
            return 0

        if value <= 50:
            return 1

    # -------- LC50 --------

    lc50 = re.search(
        r'lc50\s*[=><>]*\s*(\d+\.?\d*)',
        t
    )

    if lc50:

        value = float(
            lc50.group(1)
        )

        if value >= 200:
            return 0

        if value <= 100:
            return 1

    # -------- textual rules --------

    safe_keywords = [
        "no hemolytic activity",
        "non-hemolytic",
        "non-toxic",
        "not active",
        "absent or minimal hemolysis",
        "no signs of toxicity",
        "does not induce substantial hemolysis"
    ]

    for k in safe_keywords:
        if k in t:
            return 0

    toxic_keywords = [
        "strong hemolytic activity",
        "has hemolytic activity",
        "show hemolytic activity"
    ]

    for k in toxic_keywords:
        if k in t:
            return 1

    return np.nan

In [10]:
hemo_df["hemo_label"] = (
    hemo_df["Hemolytic_activity"]
    .apply(final_hemo_label)
)

print(
    hemo_df["hemo_label"]
    .value_counts(dropna=False)
)

hemo_label
0.0    942
1.0    567
NaN    520
Name: count, dtype: int64


In [11]:
hemo_final = hemo_df[
    hemo_df["hemo_label"].notna()
].copy()

print(hemo_final.shape)

print(
    hemo_final["hemo_label"]
    .value_counts()
)

(1509, 8)
hemo_label
0.0    942
1.0    567
Name: count, dtype: int64


In [12]:
hemo_final.to_csv(
    "hemolysis_dataset.csv",
    index=False
)

print("Saved")

Saved
